# Week 1 — Images as Arrays (STARTER)
### The Computer Vision Internship | VisionAI Lagos

**This week's question:** What does the computer actually see? Before any model — understand the data as numbers.

**Before any code — write your observations:**

## My Visual Observations

*Open five images per class before running any code. Write one sentence per class.*

**Residential:** [WRITE HERE]
**Commercial:** [WRITE HERE]
**Industrial:** [WRITE HERE]
**Green space:** [WRITE HERE]

**Which two classes will a model most likely confuse, and why?** [WRITE HERE]

## Part 1 — Images as Numerical Distributions

> ⏸️ **Pause and Predict**
> Before plotting: which channel (R/G/B) do you expect to have the highest mean across the whole dataset? Which land-use class will pull that channel highest?

In [ ]:
# Sample 200 images — compute per-channel statistics
sample = df.sample(200, random_state=42)
all_r, all_g, all_b = [], [], []
for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Loading"):
    arr = np.array(Image.open(IMG_DIR/row["filename"]))
    all_r.append(arr[:,:,0].flatten())
    all_g.append(arr[:,:,1].flatten())
    all_b.append(arr[:,:,2].flatten())
all_r = np.concatenate(all_r)
all_g = np.concatenate(all_g)
all_b = np.concatenate(all_b)

fig, axes = plt.subplots(1,3,figsize=(14,4),sharey=True)
for ax,vals,name,colour in zip(axes,[all_r,all_g,all_b],
    ["Red","Green","Blue"],["#C0392B","#27AE60","#2980B9"]):
    ax.hist(vals,bins=60,color=colour,alpha=0.8,edgecolor="none")
    ax.axvline(vals.mean(),color="black",linewidth=1.5,linestyle="--",
               label=f"mean={vals.mean():.0f}")
    ax.set_title(f"{name} channel",fontweight="bold")
    ax.set_xlabel("Pixel value (0–255)"); ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
axes[0].set_ylabel("Frequency")
plt.suptitle("Pixel Value Distributions (n=200)",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/pixel_distributions.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()
print(f"R mean={all_r.mean():.1f}  G mean={all_g.mean():.1f}  B mean={all_b.mean():.1f}")

## Part 2 — Colour Spaces and NDVI

> 🧠 **What Will This Output?**
> The NDVI proxy formula is (G-R)/(G+R). For a pure vegetation patch with G=150 and R=60, compute NDVI manually. For an industrial patch with G=110 and R=140, compute it. Which is positive?

In [ ]:
# NDVI proxy visualisation per class
examples = {}
for lu in CLASSES:
    row = df[df["land_use"]==lu].iloc[0]
    examples[lu] = np.array(Image.open(IMG_DIR/row["filename"]))

fig, axes = plt.subplots(4,4,figsize=(14,12))
for ri,(lu,arr) in enumerate(examples.items()):
    axes[ri][0].imshow(arr); axes[ri][0].set_ylabel(lu,fontweight="bold",fontsize=9); axes[ri][0].axis("off")
    grey = 0.299*arr[:,:,0]+0.587*arr[:,:,1]+0.114*arr[:,:,2]
    axes[ri][1].imshow(grey,cmap="gray",vmin=0,vmax=255); axes[ri][1].axis("off")
    axes[ri][2].imshow(arr[:,:,1],cmap="Greens",vmin=0,vmax=255); axes[ri][2].axis("off")
    g=arr[:,:,1].astype(float); r=arr[:,:,0].astype(float)
    ndvi=(g-r)/(g+r+1e-6)
    im=axes[ri][3].imshow(ndvi,cmap="RdYlGn",vmin=-0.5,vmax=0.5); axes[ri][3].axis("off")
    if ri==0:
        for ci,t in enumerate(["RGB","Greyscale","Green channel","NDVI proxy"]):
            axes[0][ci].set_title(t,fontweight="bold",fontsize=9)
plt.suptitle("Colour Space Comparison by Land-Use Class",fontweight="bold",y=1.01)
plt.tight_layout(); plt.savefig("outputs/colour_spaces.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

## Part 3 — City Spectral Signatures

> ⏸️ **Pause and Predict**
> Before computing: which city will have the highest mean brightness? Write your prediction and reasoning.

In [ ]:
# Mean brightness and NDVI by city and class
records = []
for city in df["city"].unique():
    for lu in df["land_use"].unique():
        cell = df[(df["city"]==city)&(df["land_use"]==lu)].sample(min(30,len(df[(df["city"]==city)&(df["land_use"]==lu)])),random_state=42)
        for _,row in cell.iterrows():
            arr=np.array(Image.open(IMG_DIR/row["filename"]))
            g=arr[:,:,1].astype(float); r=arr[:,:,0].astype(float)
            records.append({"city":city,"land_use":lu,"brightness":arr.mean(),
                            "ndvi_proxy":float((g-r).mean()/(g+r+1e-6).mean())})
stats = pd.DataFrame(records)

fig, axes = plt.subplots(1,2,figsize=(13,5))
city_order = stats.groupby("city")["brightness"].mean().sort_values(ascending=False).index
city_means = stats.groupby("city")["brightness"].mean().reindex(city_order)
city_stds  = stats.groupby("city")["brightness"].std().reindex(city_order)
axes[0].barh(city_order,city_means,xerr=city_stds,
             color=[CITY_CLR[c] for c in city_order],height=0.6,capsize=4,error_kw={"elinewidth":1.5})
axes[0].set_title("Kano Is Systematically Brighter — Arid Terrain Reflects More Light",
                   fontweight="bold",loc="left"); axes[0].set_xlabel("Mean pixel brightness")
axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)

lu_order = stats.groupby("land_use")["ndvi_proxy"].mean().sort_values(ascending=False).index
lu_means  = stats.groupby("land_use")["ndvi_proxy"].mean().reindex(lu_order)
lu_stds   = stats.groupby("land_use")["ndvi_proxy"].std().reindex(lu_order)
axes[1].barh(lu_order,lu_means,xerr=lu_stds,
             color=[CLASS_CLR[l] for l in lu_order],height=0.6,capsize=4,error_kw={"elinewidth":1.5})
axes[1].axvline(0,color="black",linewidth=0.8,linestyle="--")
axes[1].set_title("Green Space Has Positive NDVI; Industrial and Commercial Are Negative",
                   fontweight="bold",loc="left"); axes[1].set_xlabel("NDVI proxy (G-R)/(G+R)")
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)
plt.tight_layout(); plt.savefig("outputs/city_signatures.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

## Part 4 — GLCM Texture Features

> 🧠 **Synthesis**
> Before computing GLCM: predict which class will have the highest energy (most uniform texture). Then: write three sentences connecting this week's three feature types (spectral stats, colour spaces, GLCM). What does each capture that the others miss? Both answers should point to the same conclusion about why the CNN (Week 5) is needed.

In [ ]:
from skimage.feature import graycomatrix, graycoprops
from skimage.color import rgb2gray

def extract_glcm(arr_rgb, distances=[1,3], angles=[0,np.pi/4,np.pi/2]):
    """Extract GLCM texture features: contrast, correlation, energy, homogeneity."""
    grey = (rgb2gray(arr_rgb)*255).astype(np.uint8)
    glcm = graycomatrix(grey,distances=distances,angles=angles,
                         levels=256,symmetric=True,normed=True)
    return [graycoprops(glcm,p).mean() for p in ["contrast","correlation","energy","homogeneity"]]

print("Extracting GLCM features (sample n=400)...")
sample = df.sample(400,random_state=42)
glcm_records = []
for _,row in tqdm(sample.iterrows(),total=len(sample)):
    arr = np.array(Image.open(IMG_DIR/row["filename"]))
    feats = extract_glcm(arr)
    glcm_records.append({"city":row["city"],"land_use":row["land_use"],
                          "contrast":feats[0],"correlation":feats[1],
                          "energy":feats[2],"homogeneity":feats[3]})
glcm_df = pd.DataFrame(glcm_records)

fig,axes=plt.subplots(2,2,figsize=(13,10))
for ax,feat in zip(axes.flatten(),["contrast","correlation","energy","homogeneity"]):
    for lu,grp in glcm_df.groupby("land_use"):
        ax.hist(grp[feat],bins=30,alpha=0.55,label=lu,color=CLASS_CLR[lu],density=True)
    ax.set_title(f"GLCM {feat.title()} by Class",fontweight="bold",loc="left")
    ax.set_xlabel(feat.title()); ax.legend(fontsize=8)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.suptitle("GLCM Texture Features — Can Spatial Patterns Separate Land-Use Classes?",fontweight="bold")
plt.tight_layout(); plt.savefig("outputs/glcm_features.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()
print("\nGLCM summary by class:")
print(glcm_df.groupby("land_use")[["contrast","energy","homogeneity"]].mean().round(4))

## Part 5 — Spectral + GLCM Baseline Classifier

In [ ]:
# Build feature matrix: spectral stats + GLCM
X_spec  = sample[["mean_r","mean_g","mean_b","brightness","ndvi_proxy"]].values
# YOUR CODE HERE: merge glcm_df features with sample
# Hint: glcm_df was built from the same sample in the same order
X_glcm = glcm_df[["contrast","correlation","energy","homogeneity"]].values
X_combined = np.hstack([X_spec, X_glcm])
y = sample["land_use"].values

X_tr,X_te,y_tr,y_te = train_test_split(X_combined,y,test_size=0.2,random_state=42,stratify=y)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)

dummy  = DummyClassifier(strategy="most_frequent"); dummy.fit(X_tr_s[:,0:5],y_tr)
lr_spec = LogisticRegression(max_iter=1000,random_state=42); lr_spec.fit(X_tr_s[:,0:5],y_tr)
lr_full = LogisticRegression(max_iter=1000,random_state=42); lr_full.fit(X_tr_s,y_tr)

f1_dummy = f1_score(y_te,dummy.predict(X_te_s[:,0:5]),average="weighted")
f1_spec  = f1_score(y_te,lr_spec.predict(X_te_s[:,0:5]),average="weighted")
f1_full  = f1_score(y_te,lr_full.predict(X_te_s),average="weighted")

print(f"{'Classifier':<30} {'Weighted F1':>12}")
print("-"*44)
print(f"  {'Dummy (most_frequent)':<28} {f1_dummy:>12.3f}  ← floor")
print(f"  {'Spectral only (5 features)':<28} {f1_spec:>12.3f}")
print(f"  {'Spectral + GLCM (9 features)':<28} {f1_full:>12.3f}  ← GLCM adds {f1_full-f1_spec:.3f}")
print()
print("Interpretation:")
print("  This is your Week 1 baseline. The CNN (Week 5) must beat this.")
print(f"  When CNN achieves ~0.90, the honest statement is:")
print(f"  'improvement of +{0.90-f1_full:.2f} over spectral+GLCM baseline of {f1_full:.2f}'")

## ⚠️ Common Mistakes — Week 1

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Using `img.size` expecting (H,W) | Off-by-one in spatial operations | PIL `.size` returns (W,H); numpy `.shape` returns (H,W,C) |
| Computing stats on full dataset before split | Test distribution leaks | Sample from training split only |
| Inconsistent GLCM levels across patches | Non-comparable values | Fix `levels=256` and `normed=True` |
| Treating NDVI proxy as true NDVI | Over-claiming spectral significance | Note that true NDVI requires NIR band |

## 🏆 Challenge Task

> Extend the feature matrix with three additional features:
> 1. Standard deviation of the NDVI proxy (texture of vegetation)
> 2. Ratio of red to blue (differentiates sandy Kano from humid Lagos)
> 3. Number of pixels above brightness 200 (proportion of highly reflective surfaces)
>
> Does adding these features improve the baseline F1? By how much? Write a markdown cell interpreting the result.

## ✅ What You Can Do After This Week

- Load images as numpy arrays and extract per-channel statistics
- Convert between RGB, greyscale, and NDVI proxy representations
- Compute GLCM texture features and interpret contrast, energy, homogeneity
- Explain three generations of feature extraction and why CNNs are needed
- Build and evaluate a spectral + texture baseline classifier

---
## ✅ Week 1 Complete — Next: `week2_eda_STARTER.ipynb`

---
*Add `week1_images_as_arrays_STARTER.ipynb` to your portfolio.*

*The Computer Vision Internship · VisionAI Lagos · github.com/InternshipInABook*

## ✅ By completing Week 1, you can now:

- Explain why an image is not data until you decide what to measure
- Compute pixel intensity distributions per city class and interpret them
- Apply RGB, HSV, and greyscale transformations and explain when each is useful
- Determine whether simple thresholds can separate your classes before training

📤 **GitHub:** Push week1_images_as_arrays_STARTER.ipynb. Commit: "Week 1: Pixel distributions compared, colour spaces explored"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
